# Personal M1 → M2 ingestion and normalization
In `personal_drive`, this notebook consumes the content-addressed source imported by notebook 00, requires explicit local normalization resources, and publishes checksum-verified M1/M2 runs to the private workspace. `synthetic_ci` never accesses private data.

In [ ]:
import os
import sys

in_colab = "google.colab" in sys.modules
PROFILE = os.environ.get(
    "GENOME_EVIDENCE_PROFILE", "personal_drive" if in_colab else "synthetic_ci"
)
REPOSITORY_URL = "https://github.com/jcollins-bioinfo/genome-evidence.git"
REPOSITORY_REF = os.environ.get("GENOME_EVIDENCE_GIT_REF", "main")
WORKSPACE_ROOT = os.environ.get(
    "GENOME_EVIDENCE_WORKSPACE", "/content/drive/MyDrive/genome-evidence-private"
)
SUBJECT_ID = os.environ.get("GENOME_EVIDENCE_SUBJECT_ID", "subject-0001")

In [ ]:
import importlib
import importlib.metadata
import json
import subprocess
from hashlib import sha256
from pathlib import Path

if PROFILE not in {"personal_drive", "synthetic_ci"}:
    raise ValueError("PROFILE must be personal_drive or synthetic_ci")

CHECKOUT = Path("/content/genome-evidence-src")
if PROFILE == "personal_drive":
    if "google.colab" in sys.modules:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
    if CHECKOUT.exists():
        remote = subprocess.run(
            ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
            timeout=30,
        ).stdout.strip()
        if remote != REPOSITORY_URL:
            raise RuntimeError("Unexpected checkout remote; move the checkout aside and rerun")
        dirty = subprocess.run(
            ["git", "-C", str(CHECKOUT), "status", "--porcelain"],
            check=True,
            capture_output=True,
            text=True,
            timeout=30,
        ).stdout
        if dirty:
            raise RuntimeError("Checkout is dirty; preserve or move it aside and rerun")
    else:
        subprocess.run(
            ["git", "clone", "--no-checkout", REPOSITORY_URL, str(CHECKOUT)],
            check=True,
            timeout=180,
        )
    subprocess.run(
        ["git", "-C", str(CHECKOUT), "fetch", "--force", "origin", REPOSITORY_REF],
        check=True,
        timeout=180,
    )
    RESOLVED_COMMIT = subprocess.run(
        ["git", "-C", str(CHECKOUT), "rev-parse", "--verify", "FETCH_HEAD^{commit}"],
        check=True,
        capture_output=True,
        text=True,
        timeout=30,
    ).stdout.strip()
    subprocess.run(
        ["git", "-C", str(CHECKOUT), "checkout", "--detach", RESOLVED_COMMIT],
        check=True,
        timeout=60,
    )
    previously_imported = sys.modules.get("genome_evidence")
    if previously_imported is not None:
        previous_file = Path(getattr(previously_imported, "__file__", "")).resolve()
        if not previous_file.is_relative_to(CHECKOUT.resolve()):
            raise RuntimeError(
                "genome_evidence was already imported elsewhere; restart the runtime"
            )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "-e",
            f"{CHECKOUT}[notebook]",
        ],
        check=True,
        timeout=600,
    )
    source_root = (CHECKOUT / "src").resolve()
    package_init = source_root / "genome_evidence" / "__init__.py"
    if not package_init.is_file():
        raise RuntimeError("Resolved checkout does not contain the genome_evidence package")
    source_path = str(source_root)
    if source_path not in sys.path:
        sys.path.insert(0, source_path)
    importlib.invalidate_caches()
else:
    RESOLVED_COMMIT = "installed-ci-package"

genome_evidence = importlib.import_module("genome_evidence")
PACKAGE_ORIGIN = Path(genome_evidence.__file__).resolve()
if PROFILE == "personal_drive" and not PACKAGE_ORIGIN.is_relative_to(CHECKOUT.resolve()):
    raise RuntimeError("genome_evidence import origin is outside the resolved checkout")
INSTALLED_VERSION = importlib.metadata.version("genome-evidence")
LOCK_SHA256 = (
    sha256((CHECKOUT / "uv.lock").read_bytes()).hexdigest() if PROFILE == "personal_drive" else None
)
SANITIZED_IMPORT_PATH = (
    str(PACKAGE_ORIGIN.relative_to(CHECKOUT))
    if PROFILE == "personal_drive"
    else "installed-ci-package"
)
BOOTSTRAP_STATUS = {
    "profile": PROFILE,
    "requested_ref": REPOSITORY_REF,
    "resolved_commit": RESOLVED_COMMIT,
    "version": INSTALLED_VERSION,
    "import_path": SANITIZED_IMPORT_PATH,
    "lock_sha256": LOCK_SHA256,
    "lock_equivalent": PROFILE != "personal_drive",
}
print(json.dumps(BOOTSTRAP_STATUS, sort_keys=True))

In [ ]:
from genome_evidence.workspace import WorkspaceConfig, initialize_workspace

if PROFILE == "personal_drive":
    workspace = initialize_workspace(Path(WORKSPACE_ROOT), WorkspaceConfig(subject_id=SUBJECT_ID))
else:
    assert PROFILE == "synthetic_ci"
    workspace = None

In [ ]:
from genome_evidence.workspace import run_personal_m1_m2

if PROFILE == "personal_drive":
    result = run_personal_m1_m2(workspace, SUBJECT_ID, os.environ)
    os.environ["GENOME_EVIDENCE_NORMALIZATION_RUN"] = str(result.m2_run)
    print(
        {
            "m1_run": str(result.m1_run.relative_to(workspace)),
            "m2_run": str(result.m2_run.relative_to(workspace)),
            "observations": result.observation_count,
            "mappings": result.mapping_count,
            "canonical_genotypes": result.canonical_genotype_count,
        }
    )
else:
    assert PROFILE == "synthetic_ci"
    print("synthetic_ci: personal M1/M2 path loaded; fabricated execution is covered by tests")